# Denoising Autoencoder Detector

### Import module

In [1]:
import pandas
import yaml
import torch
import raha
import pickle

from modules.Preprocessing import Preprocessing
from modules.Trainer import DAETrainer
from modules.Detector import Detector

from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### Load Dataset

In [2]:
yaml_path = "dicts_full.yaml"
datasets = {}

with open(yaml_path, "rb") as file:
  dictionaries = yaml.safe_load(file)

for name, dict in dictionaries.items():
  datasets[name] = {
    "raw_clean": pandas.read_csv(dict["clean_path"]),
    "raw_dirty": pandas.read_csv(dict["path"]),
  }
  
  print(f"{name} → clean: {datasets[name]['raw_clean'].shape}, dirty: {datasets[name]['raw_dirty'].shape}")

beers → clean: (2410, 11), dirty: (2410, 11)
bike → clean: (17379, 17), dirty: (17379, 17)
nasa → clean: (1503, 6), dirty: (1503, 6)


### Preprocessing

In [3]:
for name, df_dict in datasets.items():
  df_clean = df_dict["raw_clean"].copy()
  df_dirty = df_dict["raw_dirty"].copy()

  num_cols = df_clean.select_dtypes(include="number").columns.tolist()
  cat_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()

  prep = Preprocessing(num_cols, cat_cols)
  df_clean = prep.fill_missing(df_clean)
  X_transformed = prep.fit_transform(df_clean)
  X_tensor = torch.tensor(X_transformed)
  loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)

  datasets[name] = {
    "df_clean": df_clean,
    "df_dirty": df_dirty,
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "prep": prep,
    "X_tensor": X_tensor,
    "loader": loader,
  }

  print(f"\n{name}")
  print(f"Total dim after preprocessing: {datasets[name]['prep'].total_dim}")
  print(f"Num cols: {datasets[name]['num_cols']}")
  print(f"Cat cols: {datasets[name]['cat_cols']}")


beers
Total dim after preprocessing: 3395
Num cols: ['index', 'id', 'ounces', 'abv', 'ibu', 'brewery_id']
Cat cols: ['beer_name', 'style', 'brewery_name', 'city', 'state']

bike
Total dim after preprocessing: 17
Num cols: ['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
Cat cols: []

nasa
Total dim after preprocessing: 6
Num cols: ['frequency', 'angle', 'chord_length', 'velocity', 'thickness', 'sound_pressure_level']
Cat cols: []


### Training

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for name, ds in datasets.items():
  print(f"Training: {name}")

  trainer = DAETrainer(
    prep = ds["prep"],
    device = device,
  )

  df_history = trainer.fit(ds["loader"])
  datasets[name]["trainer"] = trainer
  datasets[name]["df_history"] = df_history
  

Training: beers
Epoch  10/100 | Loss: 3.5747 | MSE: 0.4557 | NLL: 2.8348
Epoch  20/100 | Loss: 2.7597 | MSE: 0.3700 | NLL: 2.4671
Epoch  30/100 | Loss: 2.3424 | MSE: 0.2995 | NLL: 2.0502
Epoch  40/100 | Loss: 2.1721 | MSE: 0.3125 | NLL: 1.9585
Epoch  50/100 | Loss: 2.0918 | MSE: 0.3373 | NLL: 1.8843
Epoch  60/100 | Loss: 2.0442 | MSE: 0.3329 | NLL: 1.6565
Epoch  70/100 | Loss: 2.0158 | MSE: 0.3998 | NLL: 1.9574
Early stopping epoch 77 | Best loss: 1.9846 at epoch 67
Model restored to epoch 67
Training: bike
Epoch  10/100 | Loss: 0.2563 | MSE: 0.2874 | NLL: 0.0000
Epoch  20/100 | Loss: 0.2221 | MSE: 0.1390 | NLL: 0.0000
Epoch  30/100 | Loss: 0.2085 | MSE: 0.2243 | NLL: 0.0000
Epoch  40/100 | Loss: 0.1969 | MSE: 0.2123 | NLL: 0.0000
Epoch  50/100 | Loss: 0.1962 | MSE: 0.2329 | NLL: 0.0000
Epoch  60/100 | Loss: 0.1885 | MSE: 0.2079 | NLL: 0.0000
Epoch  70/100 | Loss: 0.1922 | MSE: 0.2842 | NLL: 0.0000
Early stopping epoch 74 | Best loss: 0.1869 at epoch 64
Model restored to epoch 64
Train

### DAE Detector

In [5]:
detectors = {}

for name, ds in datasets.items():
  print(f"Detecting: {name}")

  detector = Detector(
    trainer = ds["trainer"],
    prep = ds["prep"],
  )

  detector.fit_clean(ds["X_tensor"])
  detector.detect(ds["df_dirty"], ds["df_clean"])
  detector.summary()
  
  detectors[name] = detector

Detecting: beers
Detecting: bike
Detecting: nasa


### Output for Baran (Corrector)

In [6]:
dae_result_path = "./results/detection/DAE"

for name, dict in dictionaries.items():
  d = raha.dataset.Dataset(dict)
  d.detected_cells = detectors[name].get_detected_cells(datasets[name]["df_dirty"])

  # save output
  with open(f"{dae_result_path}/{name}.pkl", "wb") as f:
    pickle.dump(d, f)

  print(f"{name}: {len(d.detected_cells)} detected cells saved")

beers: 2326 detected cells saved
bike: 196372 detected cells saved
nasa: 2165 detected cells saved
